# 2. Normalization strategies for single-cell RNA-seq

After quality control, ambient RNA correction, and doublet detection, the next key step is **normalization**. Normalization makes expression values comparable across cells by accounting for differences in sequencing depth and other technical factors, while preserving genuine biological variation.

There is no single universally optimal normalization method; different approaches emphasize different aspects of the data. In this notebook, we compare three commonly used strategies on a Klf13 knockout embryonic mouse brain sample:

- **shifted logarithm** normalization (library-size scaling + log1p),
- normalization with size factors estimated by **scran** (R), and
- **analytic Pearson residuals**, which stabilize variance under a negative binomial model.

All methods are applied to the same QC-filtered singlet dataset, and we visualize the distribution of per-cell totals under each approach. The resulting AnnData object contains multiple normalized layers that can be used in downstream analyses (e.g. highly variable gene selection, dimensionality reduction, clustering).

## 2.1. Imports and global settings

In [ ]:
import logging
import anndata2ri
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from scipy.sparse import issparse, csr_matrix

# Set verbosity and figure parameters
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

# Suppress R callbacks
rcb.logger.setLevel(logging.ERROR)

# Activate R-to-Python conversion
ro.pandas2ri.activate()
anndata2ri.activate()

## 2.2. Configuration and input data

We start from the QC-filtered, SoupX-corrected, singlet-only AnnData object produced in the previous notebook. This object contains raw counts in `.X` and per-cell QC metrics in `.obs`.

Here we specify the sample identifier, build the input and output file paths, and load the `.h5ad` file corresponding to one Klf13 knockout sample.

In [ ]:
# Sample identifier (used to construct file paths)
sample_id = "<sample>" # e.g.96_6_M_KO_edda

# File paths
input_file = f"{sample_id}_filtered_qc_singlets.h5ad"
output_file = f"{sample_id}_qc_norm.h5ad"

print(f"Reading input AnnData: {input_file}")
adata = sc.read(filename=input_file)
adata

## 2.3. Raw library size distribution

Before normalizing, we inspect the distribution of total counts per cell. Large variation in library sizes is typical in droplet-based single-cell data and motivates the need for normalization.

In [ ]:
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False)
plt.title("Raw Counts Distribution")
plt.xlabel("Total counts per cell")
plt.ylabel("Number of cells")
plt.show()

## 2.4. Shifted logarithm normalization

A simple and widely used approach is **library-size scaling followed by a log1p transform**, sometimes called shifted logarithm normalization. Each cell’s counts are scaled such that total counts are comparable across cells, and then a logarithm is applied to compress the dynamic range and reduce the influence of extreme values.

Here, we apply Scanpy’s `normalize_total` (without fixing a target sum) and store the log1p-transformed values in a dedicated layer. This provides a baseline normalization that is easy to interpret and is often suitable for exploratory analyses.

In [ ]:
# Shifted logarithm normalization
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)
logging.info("Applied shifted logarithm normalization")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts (raw)")
axes[0].set_xlabel("Total counts per cell")

sns.histplot(adata.layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("Shifted logarithm (sum over genes)")
axes[1].set_xlabel("Sum of log1p-normalized counts")

plt.tight_layout()
plt.show()

## 2.5. Scran size-factor normalization

The **scran** method estimates cell-specific size factors using a deconvolution strategy that pools cells to obtain more stable estimates, particularly useful when there are heterogeneous populations and varying library sizes.

We use the following procedure:

1. Create a temporary preprocessed copy (`adata_pp`) for clustering.
2. Apply simple normalization and log1p transformation.
3. Perform PCA, build a neighbor graph, and run Leiden clustering to obtain coarse groups.
4. Pass the counts matrix and these clusters to `scran::computeSumFactors` in R to estimate size factors.
5. Divide the original counts by the size factors and apply log1p, storing the result in a new layer.

This approach aims to remove global depth differences while respecting heterogeneity across cell populations.

In [ ]:
# Load scran and parallel backend in R
ro.r(
    """
    library(scran)
    library(BiocParallel)
"""
)

# Preliminary clustering on a copy for scran
adata_pp = adata.copy()
sc.pp.normalize_total(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, n_comps=15)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="groups")

data_mat = adata_pp.X.T
if issparse(data_mat):
    if data_mat.nnz > 2**31 - 1:
        data_mat = data_mat.tocoo()
    else:
        data_mat = data_mat.tocsc()

ro.globalenv["data_mat"] = data_mat
ro.globalenv["input_groups"] = adata_pp.obs["groups"]
del adata_pp

# Run scran deconvolution in R
ro.r(
    """
    size_factors = sizeFactors(
        computeSumFactors(
            SingleCellExperiment(list(counts = data_mat)),
            clusters = input_groups,
            min.mean = 0.1,
            BPPARAM = MulticoreParam()
        )
    )
"""
)

# Bring size factors back to Python
adata.obs["size_factors"] = ro.r("size_factors")

# Apply size-factor normalization and log1p
scran = adata.X / adata.obs["size_factors"].values[:, None]
scran_log1p = np.log1p(scran)
adata.layers["scran_normalization"] = csr_matrix(scran_log1p)
logging.info("Applied scran normalization")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts (raw)")
axes[0].set_xlabel("Total counts per cell")

sns.histplot(adata.layers["scran_normalization"].sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("log1p with scran size factors")
axes[1].set_xlabel("Sum of scran-normalized log1p counts")

plt.tight_layout()
plt.show()

## 2.6. Analytic Pearson residuals

An alternative normalization approach is to model counts with a negative binomial distribution and compute **analytic Pearson residuals**. This method jointly performs normalization and variance stabilization by subtracting the expected mean and scaling by the expected variance under the model.

In Scanpy, `experimental.pp.normalize_pearson_residuals` implements this idea, producing a transformed matrix where residuals emphasize genes and cells with deviations from the expected technical background.

Here, we compute analytic Pearson residuals on the count matrix and store them as another layer in the AnnData object.

In [ ]:
analytic_pearson = sc.experimental.pp.normalize_pearson_residuals(adata, inplace=False)
adata.layers["analytic_pearson_residuals"] = csr_matrix(analytic_pearson["X"])
logging.info("Applied analytic Pearson residuals normalization")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts (raw)")
axes[0].set_xlabel("Total counts per cell")

analytic_sum = np.array(adata.layers["analytic_pearson_residuals"].sum(1)).flatten()
sns.histplot(analytic_sum, bins=100, kde=False, ax=axes[1])
axes[1].set_title("Analytic Pearson residuals (sum over genes)")
axes[1].set_xlabel("Sum of residuals")

plt.tight_layout()
plt.show()

## 2.7. Output and next steps

This notebook attaches three different normalized representations to the same AnnData object:

- `adata.layers["log1p_norm"]`: library-size scaled log1p-normalized counts,
- `adata.layers["scran_normalization"]`: counts normalized with scran-estimated size factors and log1p-transformed,
- `adata.layers["analytic_pearson_residuals"]`: analytic Pearson residuals.

We save this AnnData object to disk so that downstream notebooks (e.g. feature selection, dimensionality reduction, clustering) can experiment with different normalization choices while keeping QC and metadata consistent.

In [ ]:
adata.write(output_file)
logging.info(f"Saved normalized AnnData object to {output_file}")